In [21]:
import pandas as pd 
import numpy as np

print("Lets  start")

Lets  start


In [22]:
knowledge_base_clean = pd.read_csv("../data/knowledge_base_cleaned.csv")

print("Shape:",knowledge_base_clean.shape)
print("\nSeverity value counts :\n")
print(knowledge_base_clean['severity'].value_counts())


Shape: (29161, 9)

Severity value counts :

severity
normal      15110
Major        5775
Minor        2171
major        1689
critical      964
minor         919
Critical      575
trivial       456
blocker       408
Trivial       403
Unknown       353
Blocker       338
Name: count, dtype: int64


In [24]:
knowledge_base_clean.head()

,bug_id,title,description,severity,resolution,status,source_dataset,stack_trace,product_component
0,BUGZILLA-294734,Emergency 2.16.10 Release,2.16.9 is broken -- many users can't enter bug...,blocker,fixed,resolved,Mozilla,NaN,BUGZILLA
1,OTHER_APPLICATIONS-363323,DOM View is really inefficient with setting wh...,From comment in url: Current code: menuitem ->...,normal,fixed,resolved,Mozilla,NaN,OTHER_APPLICATIONS
2,SUPPORT.MOZILLA.ORG-398246,Add support for custom cookies and cache headers,Adding support for custom headers and cookie n...,blocker,fixed,resolved,Mozilla,NaN,SUPPORT.MOZILLA.ORG
3,OTHER_APPLICATIONS-318859,DCC functionality in ChatZilla isn't functional.,User-Agent: Mozilla/5.0 (Macintosh U PPC Mac O...,normal,fixed,resolved,Mozilla,NaN,OTHER_APPLICATIONS
4,DEVELOPER.MOZILLA.ORG-416840,Fix and cruft,Since we removed the breadcrumbs and title-ove...,normal,fixed,resolved,Mozilla,NaN,DEVELOPER.MOZILLA.ORG


In [23]:
knowledge_base_clean['severity'] = knowledge_base_clean['severity'].str.lower()
print(knowledge_base_clean['severity'].value_counts())


severity
normal      15110
major        7464
minor        3090
critical     1539
trivial       859
blocker       746
unknown       353
Name: count, dtype: int64


In [7]:
severity_mapping = {
    'blocker':'Critical',
'critical':'Critical',
'major':'High',
'normal':'Medium',
'minor':'Low',
'trivial':'Low',
'unknown':'Medium',

}

knowledge_base_clean['severity_mapped'] = knowledge_base_clean['severity'].map(severity_mapping)
print(knowledge_base_clean['severity_mapped'].value_counts())

severity_mapped
Medium      15463
High         7464
Low          3949
Critical     2285
Name: count, dtype: int64


In [8]:
knowledge_base_clean.to_csv("../data/knowledge_base_with_severity.csv", index = False)
print("Saved")

Saved


# New plan 

In [1]:
import sys
!{sys.executable}  -m pip install groq python-dotenv

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os

print("Current notebook working directory:", os.getcwd())
print("\nLooking for .env in parent folder...")
print("Exists:", os.path.exists("../.env"))

Current notebook working directory: C:\Users\famil\OneDrive\Desktop\AI_Smart_Bug_Analyzer_-_Fix-_Advisor-\notebooks

Looking for .env in parent folder...
Exists: False


In [17]:
env_content = "API_KEY_NAME= API_KEY"
with open  ("../.env",'w') as f:
    f.write(env_content)
print("Created .env file ")

Created .env file 


In [18]:
import os
print("Exists:", os.path.exists("../.env"))

Exists: True


In [20]:
from dotenv import load_dotenv
import os 

load_dotenv("../.env",override=True)TI

api_key = os.getenv("bug-analyzer-triage")

if api_key:
    print("API loaded successfully . Starrts with :",api_key[:8] , "...")
else:
    print("API key not found - check your .env file location/content")

API loaded successfully . Starrts with : gsk_mpz3 ...


In [32]:
knowledge_base_clean = pd.read_csv("../data/knowledge_base_with_severity.csv")

print(knowledge_base_clean.columns.tolist())

['bug_id', 'title', 'description', 'severity', 'resolution', 'status', 'source_dataset', 'stack_trace', 'product_component', 'severity_mapped']


In [42]:
from groq import Groq
import json

client  =  Groq (api_key=api_key)

# system_prompt ="""
# You are a bug triage assistant. You are given the title, description, and stack trace of a submitted bug, and you classify it by severity, priority, and affected component, with confidence scoring and reasoning.

# Reply ONLY in valid JSON, with this exact structure:
# {"bug_id": "...", "severity": "Critical", "confidence": 0.85, "reasoning": "...", "priority": "...", "component": "Login"}

# Field definitions:
# - bug_id: if provided, use it; otherwise use "unknown"
# - severity: must be exactly one of Critical, High, Medium, Low — no other words
# - confidence: a score from 0 to 1 representing how confident you are in this classification
# - reasoning: a brief explanation of why you classified it this way
# - priority: how urgently this needs to be addressed — Immediate, High, Medium, or Low (related to but different from severity: severity is impact, priority is urgency)
# - component: which part/module of the software seems affected, inferred from the content (e.g. "Login", "Database", "UI/Rendering") — do not restrict to a fixed list

# If you encounter other severity-like words, map them using this table:
# blocker/critical -> Critical, major -> High, normal/unknown -> Medium, minor/trivial -> Low


# """


system_prompt = """You are a bug triage assistant. You are given the title, description, and stack trace of a submitted bug, and you classify it by severity, priority, and affected component, with confidence scoring and reasoning.

IMPORTANT CALIBRATION: Most everyday bugs are Medium severity, not Critical or High. Reserve Critical/High for bugs that cause crashes, data loss, security issues, or completely block core functionality. Reserve Low for cosmetic issues, minor inconveniences, or feature requests. The MAJORITY of real-world bugs are routine Medium severity — do not over-classify ordinary bugs as urgent just because they mention words like "error" or "exception."

Examples for calibration:
- "App crashes on startup, no workaround" -> Critical
- "Login fails for all users" -> Critical  
- "Button text is misaligned by 2px" -> Low
- "Feature request: add dark mode" -> Low
- "Export function occasionally produces incorrect date format" -> Medium
- "Search results are slow to load under heavy traffic" -> Medium

Reply ONLY in valid JSON, with this exact structure:
{"bug_id": "...", "severity": "Critical", "confidence": 0.85, "reasoning": "...", "priority": "...", "component": "Login"}

Field definitions:
- bug_id: if provided, use it; otherwise use "unknown"
- severity: must be exactly one of Critical, High, Medium, Low — no other words
- confidence: a score from 0 to 1 representing how confident you are in this classification
- reasoning: a brief explanation of why you classified it this way
- priority: how urgently this needs to be addressed — Immediate, High, Medium, or Low (related to but different from severity: severity is impact, priority is urgency)
- component: which part/module of the software seems affected, inferred from the content (e.g. "Login", "Database", "UI/Rendering") — do not restrict to a fixed list

If you encounter other severity-like words, map them using this table:
blocker/critical -> Critical, major -> High, normal/unknown -> Medium, minor/trivial -> Low
"""


test_bug= knowledge_base_clean.iloc[0]
user_message = f"Title:{test_bug['title']} \nDescription:{test_bug['description']} "

response = client.chat.completions.create(
    model = 'llama-3.1-8b-instant',
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_message}
    ],
    temperature=0.2
)

result = response.choices[0].message.content
print(result)


{"bug_id": "unknown", "severity": "Critical", "confidence": 0.95, "reasoning": "The issue prevents many users from entering bugs, which is a core functionality. This is a clear blocker for the current release.", "priority": "Immediate", "component": "Release Management"}


In [30]:
knowledge_base_clean.columns

Index(['bug_id', 'title', 'description', 'severity', 'resolution', 'status',
       'source_dataset', 'stack_trace', 'product_component'],
      dtype='str')

In [31]:
print("Actual bug title:", test_bug['title'])
print("Actual bug description:", test_bug['description'][:200])
print("\nActual historical severity (ground truth):", test_bug['severity'])
print("\nLLM's predicted severity:", "Critical")

Actual bug title: Emergency 2.16.10 Release
Actual bug description: 2.16.9 is broken -- many users can't enter bugs on it particularly not from a fresh install. So we need to pull 2.16.9 and post a 2.16.10 instead.

Actual historical severity (ground truth): blocker

LLM's predicted severity: Critical


In [37]:
print("Actual bug title:", test_bug['title'])
print("Actual bug description:", test_bug['description'][:200])
print("\nActual historical severity (ground truth):", test_bug['severity_mapped'])
print("\nLLM's predicted severity:", "Critical")

Actual bug title: Emergency 2.16.10 Release
Actual bug description: 2.16.9 is broken -- many users can't enter bugs on it particularly not from a fresh install. So we need to pull 2.16.9 and post a 2.16.10 instead.

Actual historical severity (ground truth): Critical

LLM's predicted severity: Critical


In [39]:
import time

severities = ['Critical', 'High', 'Medium', 'Low']
sample_bugs = pd.concat([
    knowledge_base_clean[knowledge_base_clean['severity_mapped'] == s].sample(1, random_state=42)
    for s in severities
]).reset_index(drop=True)

print("Testing on", len(sample_bugs), "sample bugs (one per severity level)\n")
print(sample_bugs[['title', 'severity_mapped']])
for idx, bug in sample_bugs.iterrows():
    user_message = f"Title: {bug['title']}\nDescription: {bug['description']}\nStack Trace: Not available"

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )

    result = response.choices[0].message.content

    print(f"--- Bug {idx+1} ---")
    print("Title:", bug['title'])
    print("Actual severity:", bug['severity_mapped'])
    print("LLM response:", result)
    print()

    time.sleep(1)  # small pause to avoid hitting rate limits

Testing on 4 sample bugs (one per severity level)

                                               title severity_mapped
0  support project inheritence with flat project ...        Critical
1  Client use Axis-cpp, string array type not com...            High
2      [DND] Gtk ImageTransfer prefers JPEG over PNG          Medium
3                              IoSession.isClosing()             Low
--- Bug 1 ---
Title: support project inheritence with flat project layout
Actual severity: Critical
LLM response: {
  "bug_id": "unknown",
  "severity": "High",
  "confidence": 0.9,
  "reasoning": "The bug description mentions a specific limitation in project inheritance due to a flat directory structure, which suggests a significant impact on project functionality.",
  "priority": "High",
  "component": "Project Management"
}

--- Bug 2 ---
Title: Client use Axis-cpp, string array type not compatible with Axic at server
Actual severity: High
LLM response: {
  "bug_id": "unknown",
  "severity": "C

In [47]:
import random

random.seed(42)
test_sample = knowledge_base_clean.sample(30, random_state=42).reset_index(drop=True)

severity_order = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}

exact_matches = 0
adjacent_matches = 0
results_log = []

for idx, bug in test_sample.iterrows():
    user_message = f"Title: {bug['title']}\nDescription: {bug['description']}\nStack Trace: Not available"

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.2
        )
        result = json.loads(response.choices[0].message.content)
        predicted = result.get('severity', 'Medium')
    except Exception as e:
        predicted = 'Medium'
        print(f"Error on bug {idx}: {e}")

    actual = bug['severity_mapped']

    is_exact = (predicted == actual)
    is_adjacent = abs(severity_order.get(predicted, 1) - severity_order.get(actual, 1)) <= 1

    if is_exact:
        exact_matches += 1
    if is_adjacent:
        adjacent_matches += 1

    results_log.append({'title': bug['title'], 'actual': actual, 'predicted': predicted, 'exact': is_exact})

    time.sleep(1)

print(f"\nExact match accuracy: {exact_matches}/{len(test_sample)} = {exact_matches/len(test_sample)*100:.1f}%")
print(f"Adjacent (±1 level) accuracy: {adjacent_matches}/{len(test_sample)} = {adjacent_matches/len(test_sample)*100:.1f}%")

Error on bug 24: Expecting value: line 1 column 1 (char 0)
Error on bug 26: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxfy6ngtfr49wg8hhv46tepc` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5245, Requested 768. Please try again in 130ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Exact match accuracy: 18/30 = 60.0%
Adjacent (±1 level) accuracy: 25/30 = 83.3%


In [48]:
results_df = pd.DataFrame(results_log)
print("Prediction distribution:")
print(results_df['predicted'].value_counts())
print("\nActual distribution (in this sample):")
print(results_df['actual'].value_counts())

print("\n--- Biggest misses (predicted far from actual) ---")
severity_order = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
results_df['distance'] = results_df.apply(
    lambda r: abs(severity_order.get(r['predicted'], 1) - severity_order.get(r['actual'], 1)), axis=1
)
print(results_df[results_df['distance'] >= 2][['title', 'actual', 'predicted']])

Prediction distribution:
predicted
Medium      22
Low          4
Critical     4
Name: count, dtype: int64

Actual distribution (in this sample):
actual
Medium      19
Low          5
Critical     3
High         3
Name: count, dtype: int64

--- Biggest misses (predicted far from actual) ---
                                                title    actual predicted
0          Add Mockito and AssertJ in target platform  Critical       Low
7   SWTException: Invalid thread access on disconn...       Low  Critical
10  Access Error trying to access vpg when creatin...    Medium  Critical
19  std::numeric_limits ::traps = false when integ...       Low  Critical
21             missing metadata errors on the console  Critical    Medium


In [49]:
import time

def call_llm_with_retry(system_prompt, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = (attempt + 1) * 3  # wait 3s, then 6s, then 9s
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Non-rate-limit error: {e}")
                break
    return None  # all retries failed

In [50]:
def triage_agent(title, description, stack_trace="Not available", bug_id="unknown"):
    user_message = f"Title: {title}\nDescription: {description}\nStack Trace: {stack_trace}"
    
    result = call_llm_with_retry(system_prompt, user_message)
    
    if result is None:
        result = {"severity": "Medium", "confidence": 0.0, "reasoning": "LLM call failed after retries", "priority": "Medium", "component": "Unknown"}
    
    result['bug_id'] = bug_id
    return result

# Quick test
test_result = triage_agent(
    title="App crashes on login",
    description="Users report the app crashes immediately after entering valid credentials.",
    bug_id="TEST-001"
)
print(test_result)

{'bug_id': 'TEST-001', 'severity': 'Critical', 'confidence': 0.95, 'reasoning': 'The title explicitly states the app crashes, which is a clear indicator of a Critical severity issue. The description also confirms that the crash occurs after entering valid credentials, further supporting this classification.', 'priority': 'Immediate', 'component': 'Login'}


In [51]:
results_df.to_csv("../data/triage_validation_results.csv", index=False)
print("Saved validation results")

Saved validation results
